# Phase 1 — PLL fundamentals from first principles (steps 11–24)

This phase builds the PLL from $v = A\sin\phi(t)$ upward, so every later equation has a physical meaning you can rebuild on paper. By the end you can derive $H(s)$ of any PLL, judge its stability, predict its steady-state errors, and turn a phase-noise plot into an rms-jitter number — the four things an interviewer will test first.

**How to use this notebook.** Every step follows the same loop: **What happens** (the observation) →
**How it happens** (the mechanism and the equations, in Razavi's notation) → **Why it happens** (the physical
reason) → an interactive figure → **Try it** (what to change and what to look for) → **Interview questions**
(click a question to open the answer; try to answer it aloud first).

Notation (Razavi, *RF Microelectronics* ch. 9 and *Design of CMOS PLLs*): $\phi_{in}$, $\phi_{out}$ input and output
phase; $\omega_{out} = \omega_0 + K_{VCO} V_{cont}$; $K_{PD}$ phase-detector gain; $I_p$ charge-pump current;
$R_1, C_1, C_2$ loop filter; $N$ divide ratio; $\omega_n$, $\zeta$ natural frequency and damping;
$H(s) = \Phi_{out}/\Phi_{in}$. Frequencies $f$ in Hz, angular $\omega = 2\pi f$ in rad/s.
Design numbers are the submitted proposal's: $f_{REF} = 50$ MHz, $N = 32$, $f_{out} = 1.6$ GHz, $f_c \approx 2.5$ MHz.

## Contents

- [Step 11 — Why a PLL exists](#step-11)
- [Step 12 — Phase and frequency from scratch; the VCO is a phase integrator](#step-12)
- [Step 13 — Phase detectors and their gain K_PD](#step-13)
- [Step 14 — Charge pump: current pulses, charge per cycle, average current](#step-14)
- [Step 15 — Loop filter: why a capacitor integrates current; Z(s) of C, R+C, R+C‖C2](#step-15)
- [Step 16 — Divider: why ÷N divides both frequency and phase](#step-16)
- [Step 17 — Linear phase-domain model: G(s), H(s), E(s)](#step-17)
- [Step 18 — Order and type](#step-18)
- [Step 19 — Stability: Bode, crossover, phase margin, root locus](#step-19)
- [Step 20 — Time response and steady-state error](#step-20)
- [Step 21 — Acquisition: lock range, pull-in range, capture range, cycle slips](#step-21)
- [Step 22 — Noise basics: L(f), ring-oscillator noise, jitter and σ_t from ∫L(f)](#step-22)
- [Step 23 — Noise transfer: in-loop sources low-pass and ×N², VCO high-pass; optimum bandwidth](#step-23)
- [Step 24 — Lab: behavioural PLL in Python and ngspice; check H(s) and lock](#step-24)

*14 steps, 164 interview questions.*

In [ ]:
# setup - run this cell first (Shift+Enter). Figures are interactive: drag the slider under each plot,
# hover for values, drag to zoom, double-click to reset.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sspll_learn as sl
from sspll_learn import P, show, sweep, bode, kT
np.set_printoptions(precision=4, suppress=True)
print("ready - proposal design point:", f"{P['fref']/1e6:.0f} MHz x {P['N']} = {P['f0']/1e9:.2f} GHz")

<a id='step-11'></a>
## Step 11 — Why a PLL exists

*Roadmap step 11: crystals stop at tens of MHz, chips need GHz; jitter limits ADC SNR*

### What happens
A quartz crystal is the most stable and lowest-noise frequency source available, but a fundamental-mode crystal
oscillator is practical only up to a few tens of MHz (thinner quartz for higher frequency becomes fragile; overtone
modes go higher but pull and start-up get harder). Our proposal uses a **50 MHz** crystal oscillator. The chip needs
**1.6 GHz** (and 800/400/200/100 MHz taps). A PLL multiplies the crystal frequency by $N = 32$ while *inheriting*
its long-term accuracy and, inside the loop bandwidth, much of its low noise.

### How it happens
A voltage-controlled oscillator (VCO) can run at any frequency in its range but drifts and is noisy. The PLL
divides the VCO output by $N$, compares its **phase** with the reference, and steers the VCO until the two phases
track. If the phases track, the frequencies must be equal, so
$$f_{out} = N\,f_{REF} = 32 \times 50\ \text{MHz} = 1.6\ \text{GHz}.$$

### Why the clock quality matters: jitter → SNR
Sample a sine $v(t) = A\sin(\omega_{in} t)$ at instants that are off by a random time error $\delta t$ with rms
$\sigma_t$. The voltage error is the slope times the time error:
$$\Delta v \approx \frac{dv}{dt}\,\delta t = A\,\omega_{in}\cos(\omega_{in}t)\,\delta t
\;\Rightarrow\; \overline{\Delta v^2} = \frac{A^2\omega_{in}^2\sigma_t^2}{2}.$$
Dividing the signal power $A^2/2$ by this noise power:
$$\boxed{\text{SNR}_{jitter} = -20\log_{10}\!\left(2\pi f_{in}\,\sigma_t\right)}\qquad
\text{ENOB} = \frac{\text{SNR}-1.76}{6.02}.$$
For our target $\sigma_t = 1.4$ ps and $f_{in} = 100$ MHz: $\text{SNR} = -20\log(2\pi\cdot10^8\cdot1.4\cdot10^{-12}) = 61.1$ dB,
about 9.9 bits. That is why a *jitter* number, not just a frequency, is the PLL's main specification.

In [ ]:
fin = np.logspace(6, 9.5, 300)                      # input frequency 1 MHz ... 3.2 GHz
def frame(sig_ps):
    snr = -20*np.log10(2*np.pi*fin*sig_ps*1e-12)
    s100 = -20*np.log10(2*np.pi*100e6*sig_ps*1e-12)
    return ([dict(x=fin, y=snr, name=f"sigma_t = {sig_ps} ps", mode="lines"),
             dict(x=[100e6], y=[s100], name="100 MHz input", mode="markers", marker=dict(size=10))],
            f"Jitter-limited SNR: {s100:.1f} dB = {(s100-1.76)/6.02:.1f} bits at 100 MHz for sigma_t = {sig_ps} ps")
fig = sweep([0.1, 0.2, 0.5, 0.84, 1.0, 1.4, 2.3, 3.2, 5.0, 10.0], frame, "rms jitter (ps)", active=5,
            layout=lambda f: f.update_xaxes(type="log", title="input frequency f_in (Hz)")
                              .update_yaxes(title="SNR (dB)", range=[20, 110]))
show(fig)

**Try it.** Move the slider from 1.4 ps (FFNC on, our spec) to 2.3 ps (FFNC off): the 100 MHz SNR drops by
$20\log(2.3/1.4) = 4.3$ dB — almost one bit. Every halving of jitter buys 6 dB (one bit) at every input frequency.

**Key takeaways.** (1) A PLL is a frequency multiplier with memory of phase. (2) The crystal sets accuracy; the PLL
sets how much of the crystal's quality survives the multiplication. (3) Clock jitter becomes a hard SNR ceiling
$-20\log(2\pi f_{in}\sigma_t)$ that no ADC design can beat.

### Interview questions — step 11 (15)

<details><summary><b>Q1.</b> Why not just use a 1.6 GHz crystal?</summary>

Fundamental-mode quartz resonators get thinner as frequency rises; above roughly 50–100 MHz they become fragile and hard to make. Overtone and SAW/BAW resonators reach higher, but with less pulling range, higher cost and often worse close-in noise. A PLL from a cheap low-frequency crystal is smaller, programmable and cheaper.

</details>

<details><summary><b>Q2.</b> What does a PLL inherit from the reference, and what not?</summary>

Long-term frequency accuracy exactly ($f_{out}=Nf_{REF}$ on average) and the reference phase noise inside the loop bandwidth, multiplied by $N^2$ (+20log N dB). Outside the bandwidth the output follows the VCO's own noise, not the reference.

</details>

<details><summary><b>Q3.</b> Derive the jitter-limited SNR of an ADC.</summary>

$v=A\sin\omega t$, sampling error $\delta t$ gives $\Delta v = A\omega\cos(\omega t)\,\delta t$; mean square $A^2\omega^2\sigma_t^2/2$; signal power $A^2/2$; SNR $= 1/(\omega\sigma_t)^2$ → $-20\log(2\pi f_{in}\sigma_t)$.

</details>

<details><summary><b>Q4.</b> Does the jitter SNR depend on the signal amplitude or the sampling rate?</summary>

No. Both signal and error scale with $A$, and the per-sample error does not depend on $f_s$ (for white jitter). It depends only on the input frequency and the rms jitter.

</details>

<details><summary><b>Q5.</b> How much jitter is allowed for a 12-bit ADC at 200 MHz input?</summary>

12 bits → SNR ≈ 74 dB → $\sigma_t = 10^{-74/20}/(2\pi\cdot 2\times10^8) \approx 0.16$ ps. Hence high-speed ADCs need LC PLLs or noise-cancelled ring PLLs.

</details>

<details><summary><b>Q6.</b> Which jitter is relevant for an ADC: period jitter or absolute jitter?</summary>

Absolute (accumulated, time-interval-error) jitter integrated over the band that aliases into the signal band — essentially the integrated phase noise from a low offset up to about $f_s/2$. Period jitter matters more for digital timing closure.

</details>

<details><summary><b>Q7.</b> Why is jitter given as 'integrated from 10 kHz to 40 MHz'?</summary>

Rms jitter is the integral of phase noise over an offset band. The lower limit excludes slow wander that a receiver tracks or that is outside the application's concern; the upper limit is set by the application (here roughly $f_{REF}$-related noise). Different bands give different numbers, so the band must always be stated.

</details>

<details><summary><b>Q8.</b> What sets the maximum frequency of a ring-oscillator PLL in SKY130?</summary>

The delay of one stage: $f = 1/(2N_{stages}t_d)$. With 1.8 V devices and layout wiring, a 5-stage ring reaches about 2.2 GHz at the slow corner in our study, so 1.6 GHz keeps ~35% speed margin.

</details>

<details><summary><b>Q9.</b> Why use a ring oscillator instead of an LC oscillator here?</summary>

Area (no inductor), wide tuning range, multiphase outputs, and it is a standard-cell-friendly block. The price is ~20–30 dB worse phase noise per unit power, which this project attacks with a wide loop and feedforward noise cancellation.

</details>

<details><summary><b>Q10.</b> What is the jitter figure of merit and our target?</summary>

$FoM = 10\log[(\sigma_t/1\,s)^2 (P/1\,mW)]$. With 1.4 ps and 9 mW: $20\log(1.4\times10^{-12}) + 10\log 9 = -237.1+9.5 = -227.5$ dB.

</details>

<details><summary><b>Q11.</b> If you double the power of a ring oscillator, what happens to its jitter?</summary>

Phase noise scales as $1/P$ for a fixed frequency (Hajimiri), so free-running jitter falls by $\sqrt2$ (3 dB). That is why FoM is a fair comparison: it normalises jitter to power.

</details>

<details><summary><b>Q12.</b> Name three systems where PLL jitter is the limiting factor.</summary>

ADC/DAC sampling clocks, serial-link transmitters/receivers (eye closure, BER), RF local oscillators (reciprocal mixing, EVM).

</details>

<details><summary><b>Q13.</b> What does the PLL output look like before lock?</summary>

The VCO runs at whatever frequency its control voltage gives; the phase detector produces a beat at the difference frequency, and acquisition circuits (PFD or FLL) pull the frequency until phase lock is possible.

</details>

<details><summary><b>Q14.</b> Why do we also provide /2, /4, /8, /16 outputs?</summary>

SoC blocks need several clock rates. Absolute jitter in seconds is preserved through an ideal divider, so the divided clocks carry the same jitter while phase noise in dBc/Hz drops by $20\log M$.

</details>

<details><summary><b>Q15.</b> Why is phase noise in dBc/Hz 24 dB lower at the /16 output, but the jitter the same?</summary>

Dividing by M divides phase by M (phase power by $M^2$ → −20log M dB) but also divides the frequency by M, so $\sigma_t = \sigma_\phi/(2\pi f)$ is unchanged.

</details>

<a id='step-12'></a>
## Step 12 — Phase and frequency from scratch; the VCO is a phase integrator

*Roadmap step 12: ω = dφ/dt, φ = ∫ω dt, so a VCO is K_VCO/s*

### What happens
A periodic signal $x(t) = A\cos\phi(t)$ is described completely by its **phase** $\phi(t)$, which counts cycles
(×2π). Frequency is how fast the phase advances:
$$\omega(t) = \frac{d\phi}{dt}\qquad\Longleftrightarrow\qquad \phi(t) = \int_{-\infty}^{t}\omega(\tau)\,d\tau.$$

### How it happens in a VCO
An ideal VCO maps control voltage to frequency linearly (Razavi, *RF Microelectronics*, ch. 8):
$$\omega_{out} = \omega_0 + K_{VCO}V_{cont}\quad [K_{VCO}\text{ in rad/s/V}].$$
Therefore the output phase is
$$\phi_{out}(t) = \omega_0 t + K_{VCO}\!\int V_{cont}\,dt = \omega_0 t + \phi_{ex}(t),$$
where $\phi_{ex}$ is the **excess phase** — the only part a PLL controls. Taking the Laplace transform:
$$\boxed{\frac{\Phi_{ex}(s)}{V_{cont}(s)} = \frac{K_{VCO}}{s}}$$

### Why it matters
The VCO is an **integrator** of control voltage into phase. A constant $V_{cont}$ step does not shift the phase by a
constant; it makes the phase **ramp** forever (a frequency change). This free integrator is why every PLL is at
least Type I, and why the loop can force *zero frequency error* even with a finite gain elsewhere.

In [ ]:
f0, Kv = 1e9, 2*np.pi*300e6          # 1 GHz toy VCO, 300 MHz/V (so the waveforms stay readable)
t = np.linspace(0, 20e-9, 2000)       # 20 ns
t_step = 5e-9
def frame(dV):
    vcont = np.where(t > t_step, dV, 0.0)
    f_inst = f0 + Kv/(2*np.pi)*vcont
    phi_ex = np.cumsum(Kv*vcont)*(t[1]-t[0])
    return ([dict(x=t*1e9, y=np.cos(2*np.pi*f0*t), name="VCO with V_cont = 0", line=dict(dash="dot"), row=1, col=1),
             dict(x=t*1e9, y=np.cos(2*np.pi*f0*t + phi_ex), name="VCO with step", row=1, col=1),
             dict(x=t*1e9, y=f_inst/1e9, name="frequency (GHz)", row=2, col=1),
             dict(x=t*1e9, y=phi_ex, name="excess phase (rad)", row=3, col=1)],
            f"Step of {dV*1e3:.0f} mV at 5 ns: frequency +{Kv*dV/2/np.pi/1e6:.0f} MHz, phase ramps at {Kv*dV/1e9:.2f} rad/ns")
fig = sweep([0.0, 0.05, 0.1, 0.2, 0.3, 0.5], frame, "V_cont step (V)", active=3, rows=3, cols=1,
            subplot_titles=("waveform", "instantaneous frequency (GHz)", "excess phase phi_ex (rad)"),
            shared_xaxes=True, height=650,
            layout=lambda f: f.update_xaxes(title_text="time (ns)", row=3, col=1))
show(fig)

**Try it.** With a 0.2 V step the frequency jumps by 60 MHz *instantly*, but the phase only starts to *ramp*: after
the step the stepped waveform slides further and further away from the reference waveform. Phase is the integral
of frequency.

**Key takeaways.** $\omega = d\phi/dt$; a VCO is $K_{VCO}/s$ from $V_{cont}$ to excess phase; a PLL works on
excess phase; the integrator is free and perfect (an oscillator never "forgets" a cycle).

### Interview questions — step 12 (12)

<details><summary><b>Q1.</b> Why is a VCO modelled as $K_{VCO}/s$ and not as a gain $K_{VCO}$?</summary>

Its input controls frequency, and phase is the integral of frequency. In the phase domain (the variable a PLL compares) the VCO is an ideal integrator, $\Phi_{ex}/V_{cont} = K_{VCO}/s$.

</details>

<details><summary><b>Q2.</b> What is 'excess phase'?</summary>

The deviation of the oscillator's phase from the ideal ramp $\omega_0 t$: $\phi_{ex}(t) = \phi_{out}(t) - \omega_0 t = K_{VCO}\int V_{cont}dt$. Linear PLL analysis is done entirely on excess phases.

</details>

<details><summary><b>Q3.</b> Units of $K_{VCO}$ in the equations?</summary>

rad/s/V. Datasheets give Hz/V; multiply by $2\pi$. Forgetting the $2\pi$ is the most common loop-design error.

</details>

<details><summary><b>Q4.</b> A VCO with $K_{VCO}=2\pi\cdot100$ MHz/V gets a 1 mV step. What happens to its phase after 1 µs?</summary>

Frequency changes by 100 kHz; phase grows by $2\pi\cdot10^5\cdot10^{-6} = 0.63$ rad and keeps growing linearly.

</details>

<details><summary><b>Q5.</b> Can a PLL lock with a constant frequency difference between input and VCO free-running frequency?</summary>

Yes, the loop applies whatever $V_{cont}$ is needed. In a Type-I loop that requires a constant phase error; in a Type-II loop the filter's integrator holds $V_{cont}$ with zero phase error.

</details>

<details><summary><b>Q6.</b> Is $K_{VCO}$ constant in practice?</summary>

No. In ring oscillators it varies with control voltage, process and temperature, often by 2–3×. Loop bandwidth and phase margin move with it; designs check stability at the extreme $K_{VCO}$ values.

</details>

<details><summary><b>Q7.</b> What is the instantaneous frequency of $\cos(\omega_0t + m\sin\omega_mt)$?</summary>

$\omega_0 + m\omega_m\cos\omega_m t$: peak deviation $m\omega_m$. This is how small periodic phase modulation (spurs) relates to frequency modulation.

</details>

<details><summary><b>Q8.</b> Why is $\phi_{out} = \int\omega\,dt$ an ideal integrator without leakage?</summary>

Because phase is a count of zero crossings: an oscillator cannot 'forget' a cycle it has completed. Any frequency error accumulates as phase error without bound. This is also why free-running phase noise grows as $1/f^2$ (a random walk).

</details>

<details><summary><b>Q9.</b> How does $V_{cont}$ noise appear at the output?</summary>

As phase noise $S_\phi(f) = K_{VCO}^2 S_v(f)/(2\pi f)^2$: white voltage noise becomes $1/f^2$ phase noise, filtered by the loop's high-pass $1-H$ when locked.

</details>

<details><summary><b>Q10.</b> Why does a larger $K_{VCO}$ make a PLL noisier?</summary>

Every noise voltage on $V_{cont}$ (filter resistor, CP, supply coupling) is multiplied by $K_{VCO}$. Designs use a coarse tuning code plus a small fine $K_{VCO}$ for this reason.

</details>

<details><summary><b>Q11.</b> Express the sampled phase of a 1.6 GHz VCO 32 cycles after a reference edge if it runs 1 ppm fast.</summary>

Extra phase $= 2\pi\cdot32\cdot10^{-6} = 2.0\times10^{-4}$ rad per reference period, accumulating each period unless corrected.

</details>

<details><summary><b>Q12.</b> Difference between phase modulation and frequency modulation in this picture?</summary>

PM: the modulating signal sets $\phi$ directly; FM: it sets $d\phi/dt$. A VCO is an FM modulator from $V_{cont}$; seen from phase it is PM through an integrator.

</details>

<a id='step-13'></a>
## Step 13 — Phase detectors and their gain K_PD

*Roadmap step 13: XOR, mixer, sample-and-hold, PFD; gain and linear range of each*

### What happens
A phase detector (PD) produces an output whose **average** is a function of the phase difference
$\Delta\phi = \phi_{1}-\phi_{2}$ of its inputs. Its small-signal slope $K_{PD} = d\overline{V}/d\Delta\phi$ (V/rad) or
$d\overline{I}/d\Delta\phi$ (A/rad) is the PD's contribution to loop gain.

### How each one works
| PD | Average output | $K_{PD}$ | Linear range | Detects frequency? |
|---|---|---|---|---|
| XOR (50% duty squares) | $V_{DD}\,\Delta\phi/\pi$, triangular | $V_{DD}/\pi$ | 0 … π (lock at π/2) | no |
| Mixer (sines) | $\frac{A_1A_2}{2}\cos\Delta\phi$ | $\frac{A_1A_2}{2}$ at $\Delta\phi=\pi/2$ | ±≈π/4 around π/2 | no |
| Sample-and-hold (sample a sine $A\sin\theta$ with an edge) | $A\sin\theta_e$ | $A$ (per rad of the sampled signal) | ±≈π/4 around 0 | no |
| PFD + charge pump | $I_p\,\Delta\phi/2\pi$ | $I_p/2\pi$ (A/rad) | ±2π | **yes** |

### Why they differ
XOR and mixer outputs swing at **2× the input frequency**; only their average carries phase, so a filter must remove a
large ripple. The sample-and-hold output is a DC level once locked (no ripple) — the root idea of the sub-sampling PD.
The PFD is a state machine; its output depends on *which edge came first*, so it also sees frequency differences.

In [ ]:
dphi = np.linspace(-2*np.pi, 2*np.pi, 401)
def xor_avg(dp, duty):
    th = np.linspace(0, 1, 2000, endpoint=False)
    a = (th % 1) < duty
    return np.array([np.mean(a ^ (((th - p/(2*np.pi)) % 1) < duty)) for p in dp])
def frame(duty):
    x = xor_avg(dphi, duty)
    return ([dict(x=dphi/np.pi, y=x, name=f"XOR, duty {duty:.2f} (x V_DD)", row=1, col=1),
             dict(x=dphi/np.pi, y=0.5*np.cos(dphi), name="mixer, A1A2 = 1 (V)", row=1, col=2),
             dict(x=dphi/np.pi, y=0.5*np.sin(dphi), name="S&H of a 0.5 V sine (V)", row=2, col=1),
             dict(x=dphi/np.pi, y=dphi/(2*np.pi), name="PFD+CP (x I_p)", row=2, col=2)],
            f"Average PD output vs phase difference. XOR peak = {x.max():.2f} V_DD at duty {duty:.2f}")
fig = sweep([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7], frame, "XOR input duty cycle", active=3, rows=2, cols=2,
            subplot_titles=("XOR", "mixer", "sample-and-hold", "PFD + charge pump"), height=600,
            layout=lambda f: f.update_xaxes(title_text="dphi / pi"))
show(fig)

**Try it.** Move the XOR duty cycle away from 0.5: the triangle loses its full swing and gets flat tops — the gain
and the usable range both shrink. That is why XOR detectors need 50% duty-cycle inputs (Phase 2, step 28). The PFD
characteristic is a straight line over ±2π and depends on neither duty cycle nor amplitude.

**Key takeaways.** $K_{PD}$ is the slope at the lock point. Multiplying PDs (XOR, mixer) have periodic, bounded
characteristics and ripple at $2\omega$; the S&H PD has no ripple; the PFD has the widest range and detects frequency.

### Interview questions — step 13 (14)

<details><summary><b>Q1.</b> Derive the mixer phase-detector output.</summary>

$A_1\cos\omega t\cdot A_2\cos(\omega t+\Delta\phi) = \frac{A_1A_2}{2}[\cos\Delta\phi + \cos(2\omega t+\Delta\phi)]$; the low-pass filter keeps $\frac{A_1A_2}{2}\cos\Delta\phi$. Zero output at $\Delta\phi=\pi/2$, slope $-A_1A_2/2$ there.

</details>

<details><summary><b>Q2.</b> Why does a mixer or XOR PLL lock at 90° and not at 0°?</summary>

The average output must equal the value that holds $V_{cont}$; for a symmetric output the zero-average/centre point of the characteristic is at $\pi/2$, where the slope is maximum. At 0° the slope is zero (no gain).

</details>

<details><summary><b>Q3.</b> Derive the XOR gain.</summary>

For 50% squares the output is high for a fraction $\Delta\phi/\pi$ of each half period, so the average is $V_{DD}\Delta\phi/\pi$ for $0\le\Delta\phi\le\pi$: $K_{PD}=V_{DD}/\pi$.

</details>

<details><summary><b>Q4.</b> What is the main drawback of the XOR PD?</summary>

Large ripple at $2f_{REF}$ that must be filtered (limits bandwidth, causes spurs), sensitivity to duty cycle, no frequency detection, and a static phase error in a Type-I loop.

</details>

<details><summary><b>Q5.</b> What does 'linear range' of a PD mean and why does it matter?</summary>

The range of $\Delta\phi$ over which the characteristic is close to a straight line. Outside it the loop gain falls or changes sign; large disturbances can cause cycle slips.

</details>

<details><summary><b>Q6.</b> Why does a PFD detect frequency but an XOR does not?</summary>

The PFD remembers which input edge came first: with unequal frequencies one output is active most of the time, giving a non-zero average with the right sign. The XOR output averages to $V_{DD}/2$ over a beat period.

</details>

<details><summary><b>Q7.</b> What is $K_{PD}$ of a PFD with charge pump?</summary>

$\bar I = I_p\Delta\phi/(2\pi)$, so $K_{PD} = I_p/2\pi$ in A/rad. It drives a filter impedance, not a voltage node.

</details>

<details><summary><b>Q8.</b> What limits the linear range of a PFD to ±2π?</summary>

When the phase error exceeds 2π an edge is 'skipped' (a cycle slip); the characteristic wraps. Reset delay and dead time also shave the range near ±2π.

</details>

<details><summary><b>Q9.</b> What is the gain of a sample-and-hold PD sampling $A\sin(\omega_{VCO}t)$?</summary>

A timing error $\Delta t$ gives $V = A\sin(\omega_{VCO}\Delta t) \approx A\omega_{VCO}\Delta t$; per radian of VCO phase, $K = A$ (V/rad). In seconds, $K_t = A\omega_{VCO}$ (V/s).

</details>

<details><summary><b>Q10.</b> Which PD gains depend on signal amplitude, and why is that a problem?</summary>

Mixer and sample-and-hold. Amplitude varies over PVT, so the loop gain and bandwidth move; designs either limit the amplitude or accept the spread in phase margin.

</details>

<details><summary><b>Q11.</b> What is a bang-bang (binary) PD?</summary>

It outputs only the sign of the phase error (early/late), e.g. a flip-flop. The gain is nonlinear (depends on jitter), used in CDRs and digital PLLs.

</details>

<details><summary><b>Q12.</b> Which PD is used in our SSPLL main loop, and in the FLL?</summary>

Main loop: a sub-sampling (sample-and-hold) PD that samples the 1.6 GHz output with the 50 MHz reference. FLL: a PFD with a dead zone comparing the reference with the /32 divider output.

</details>

<details><summary><b>Q13.</b> Why is the output of an XOR at lock a square wave and not DC?</summary>

Two 50% squares 90° apart produce an XOR output at $2f$ with 50% duty; only its average is $V_{DD}/2$. This $2f$ ripple modulates the VCO unless filtered.

</details>

<details><summary><b>Q14.</b> What happens to a mixer PD if one input has harmonics?</summary>

Harmonics mix with harmonics; the PLL can lock to a harmonic or subharmonic of the reference (false lock). PFDs do not have this problem.

</details>

<a id='step-14'></a>
## Step 14 — Charge pump: current pulses, charge per cycle, average current

*Roadmap step 14: I_avg = I_p·Δφ/2π*

### What happens
The PFD outputs an UP (or DN) pulse as wide as the time between the two input edges. The charge pump turns that
pulse into a current $\pm I_p$ for exactly that time, so each reference period it delivers a packet of charge.

### How
A phase error $\Delta\phi$ (rad of the reference) is a time error $\Delta t = \frac{\Delta\phi}{2\pi}T_{REF}$. The charge per
period is $Q = I_p\Delta t$, so the **average** current is
$$\overline{I} = \frac{Q}{T_{REF}} = I_p\frac{\Delta\phi}{2\pi}.$$
Into a capacitor $C_1$ the control voltage moves by (Razavi eq. 9.12)
$$\Delta V_{cont} = \frac{\Delta\phi_0}{2\pi}\,T_{REF}\,\frac{I_p}{C_1}\quad\text{every period,}$$
i.e. a staircase that rises without limit for a constant phase error. The continuous-time (CT) approximation replaces
the staircase by its average ramp: the PFD/CP/C cascade is an **integrator**
$$\frac{V_{cont}(s)}{\Delta\Phi(s)} = \frac{I_p}{2\pi C_1 s}.$$

### Why it matters
Combined with the VCO (another integrator) this gives the loop **two integrators** — Type II — and so zero static phase
error. The CT approximation is valid only when the loop changes little in one reference period ($f_c \ll f_{REF}$).

In [ ]:
Tref, Ip, C1 = 20e-9, 100e-6, 10e-12            # 50 MHz reference, 100 uA, 10 pF
t = np.linspace(0, 8*Tref, 1601)
def frame(dphi_deg):
    dt = dphi_deg/360*Tref
    up = ((t % Tref) < dt).astype(float)
    v = np.cumsum(up*Ip)*(t[1]-t[0])/C1
    v_ct = Ip*dphi_deg/360/C1*t
    return ([dict(x=t*1e9, y=up, name="UP pulse", row=1, col=1),
             dict(x=t*1e9, y=v*1e3, name="V_cont (staircase)", row=2, col=1),
             dict(x=t*1e9, y=v_ct*1e3, name="CT approximation (ramp)", line=dict(dash="dash"), row=2, col=1)],
            f"dphi = {dphi_deg} deg: pulse {dt*1e9:.2f} ns, step {Ip*dt/C1*1e3:.2f} mV per period, I_avg = {Ip*dphi_deg/360*1e6:.1f} uA")
fig = sweep([5, 10, 20, 45, 90, 180], frame, "phase error (deg)", active=2, rows=2, cols=1, shared_xaxes=True, maxpts=1601,
            subplot_titles=("PFD UP output", "control voltage on C1 (mV)"), height=560,
            layout=lambda f: f.update_xaxes(title_text="time (ns)", row=2, col=1))
show(fig)

**Try it.** Double the phase error: the pulses double in width, the step per period doubles, the average current
doubles — the PFD/CP is *linear* in $\Delta\phi$ (until ±2π). The dashed ramp is the CT approximation through the middle of
the staircase.

**Key takeaways.** $\overline I = I_p\Delta\phi/2\pi$; a charge pump into a capacitor is an integrator; the staircase vs ramp
difference is the sampled-data nature that limits the loop bandwidth to about $f_{REF}/10$ (step 37).

### Interview questions — step 14 (12)

<details><summary><b>Q1.</b> Why is a charge-pump PLL called Type II?</summary>

The CP into a capacitor integrates the phase error and the VCO integrates the control voltage: two poles at the origin in the open loop.

</details>

<details><summary><b>Q2.</b> Derive the average charge-pump current.</summary>

Pulse width $\Delta t = \Delta\phi T/2\pi$, charge $I_p\Delta t$ per period $T$: $\bar I = I_p\Delta\phi/2\pi$.

</details>

<details><summary><b>Q3.</b> Why is the PFD/CP/C cascade not strictly linear?</summary>

It is a sampled system: charge is delivered in pulses once per period. Doubling $\Delta\phi$ doubles the pulse width but the waveform is not simply scaled at every instant. The CT approximation averages it.

</details>

<details><summary><b>Q4.</b> What is the continuous-time approximation and when is it valid?</summary>

Replacing the per-cycle charge packets by their average current. Valid when the loop bandwidth is much lower than $f_{REF}$ (rule of thumb $f_c < f_{REF}/10$).

</details>

<details><summary><b>Q5.</b> What happens to $V_{cont}$ with a constant phase error and only a capacitor?</summary>

It ramps without bound (integrator), forcing the VCO frequency to change until the phase error becomes zero. That is why the steady-state phase error is zero.

</details>

<details><summary><b>Q6.</b> Why do charge pumps use current sources and not resistors?</summary>

A current source gives charge independent of the output voltage (over its compliance range), keeping $K_{PD}$ constant as $V_{cont}$ moves.

</details>

<details><summary><b>Q7.</b> What is charge-pump compliance and why does it matter?</summary>

The range of output voltage over which UP and DN currents stay equal and constant. Outside it the currents mismatch, causing static phase offset and spurs, and the VCO tuning range usable by the loop shrinks.

</details>

<details><summary><b>Q8.</b> What happens when UP and DN currents are mismatched?</summary>

At lock the net charge per period must be zero, so the PFD settles at a phase offset where the longer pulse of the weaker source balances the stronger one. The mismatch pulses each period create ripple and reference spurs.

</details>

<details><summary><b>Q9.</b> Why does the PFD output both UP and DN briefly at lock?</summary>

The reset path has a finite delay; both outputs are high for that delay every cycle. This removes the dead zone but makes mismatch currents flow every period.

</details>

<details><summary><b>Q10.</b> How much charge does our proposal's SSPLL charge pump deliver differently?</summary>

It is a pulsed transconductor: current $g_m V_{sample}$ for a short pulse $\tau$ per period, so average current $g_m V\tau/T$. It is driven by a voltage, not by a pulse width (Phase 4, step 50).

</details>

<details><summary><b>Q11.</b> Compute $\Delta V_{cont}$ per period for $\Delta\phi=10°$, $I_p=100$ µA, $C=10$ pF, $f_{REF}=50$ MHz.</summary>

$\Delta t = (10/360)\cdot20$ ns = 0.556 ns; $Q = 55.6$ fC; $\Delta V = 5.56$ mV per period.

</details>

<details><summary><b>Q12.</b> What are clock feedthrough and charge injection in a charge pump?</summary>

Switching transistors couple the control edges through their gate-drain capacitance and dump channel charge onto the output each period; both create periodic disturbances → reference spurs.

</details>

<a id='step-15'></a>
## Step 15 — Loop filter: why a capacitor integrates current; Z(s) of C, R+C, R+C‖C2

*Roadmap step 15: Bode plots of Z(s)*

### What happens
The loop filter converts the charge-pump current into the control voltage. Its transimpedance $Z(s) = V_{cont}/I$
shapes the loop gain.

### How
A capacitor obeys $i = C\,dv/dt$, so $v = \frac1C\int i\,dt$ → $Z_C = \frac{1}{sC}$: an integrator. Adding a series
resistor gives
$$Z_1(s) = R_1 + \frac{1}{sC_1} = \frac{1+sR_1C_1}{sC_1}\qquad\text{(pole at 0, zero at } \omega_z = 1/R_1C_1).$$
The current through $R_1$ makes $V_{cont}$ jump by $I_pR_1$ every time the CP turns on, so a ripple capacitor $C_2$
is added in parallel:
$$Z(s) = \frac{1+sR_1C_1}{s(C_1+C_2)\left(1+sR_1\frac{C_1C_2}{C_1+C_2}\right)},\qquad
\omega_z = \frac{1}{R_1C_1},\quad \omega_{p3} = \frac{C_1+C_2}{R_1C_1C_2}.$$

### Why
The integrator (pole at 0) gives Type II; the zero restores phase (+90° at high frequency) so the loop can be stable
(step 34); the extra pole from $C_2$ filters the ripple but takes phase back. Between $\omega_z$ and $\omega_{p3}$ the filter is
almost a resistor, $|Z|\approx R_1$ — that flat region is where the crossover frequency is placed.

In [ ]:
f = np.logspace(3, 9, 500)
R1, C1 = 18.05e3, 13.16e-12                       # the 2.5 MHz design of step 24 / 38
def frame(ratio):
    C2 = C1*ratio
    Zc  = lambda s: 1/(s*C1)
    Zrc = lambda s: R1 + 1/(s*C1)
    Z3  = lambda s: (1+s*R1*C1)/(s*(C1+C2)*(1+s*R1*C1*C2/(C1+C2)))
    out = []
    for name, Z, dash in (("C1 only", Zc, "dot"), ("R1 + C1", Zrc, "dash"), (f"R1 + C1 || C2 (C2/C1={ratio})", Z3, None)):
        m, p = bode(Z, f)
        out.append(dict(x=f, y=m, name=name, line=dict(dash=dash), row=1, col=1))
        out.append(dict(x=f, y=p, name=name+" phase", line=dict(dash=dash), showlegend=False, row=2, col=1))
    fz = 1/(2*np.pi*R1*C1); fp = (C1+C2)/(2*np.pi*R1*C1*C2)
    return out, f"zero {fz/1e6:.2f} MHz, third pole {fp/1e6:.2f} MHz, pole/zero ratio = 1 + C1/C2 = {1+1/ratio:.1f}"
fig = sweep([0.02, 0.05, 0.0774, 0.1, 0.2, 0.5], frame, "C2 / C1", fmt="{:.3g}", active=2, rows=2, cols=1,
            shared_xaxes=True, subplot_titles=("|Z| (dB Ohm)", "phase of Z (deg)"), height=620,
            layout=lambda f: f.update_xaxes(type="log").update_xaxes(title_text="frequency (Hz)", row=2, col=1))
show(fig)

**Try it.** At $C_2/C_1 = 0.077$ (our 60° design) the phase climbs from −90° back toward 0° after the zero and turns
down again at the third pole; the maximum phase sits at $\sqrt{\omega_z\omega_{p3}}$. A bigger $C_2$ pulls the pole toward the
zero and the phase bump collapses — less phase margin (step 36).

### Interview questions — step 15 (11)

<details><summary><b>Q1.</b> Why must a charge-pump loop filter contain a resistor?</summary>

With a capacitor alone the loop has two integrators and −180° phase at every frequency: zero phase margin, the loop rings or oscillates. The series R creates a zero that adds up to +90°.

</details>

<details><summary><b>Q2.</b> Write $Z(s)$ of R1 in series with C1, all in parallel with C2.</summary>

$Z = \frac{1+sR_1C_1}{s(C_1+C_2)(1+sR_1C_s)}$ with $C_s = C_1C_2/(C_1+C_2)$.

</details>

<details><summary><b>Q3.</b> Why add C2?</summary>

Each CP pulse makes $V_{cont}$ jump by $I_pR_1$; that step modulates the VCO and makes large reference spurs. C2 smooths it, at the cost of a third pole (less phase margin).

</details>

<details><summary><b>Q4.</b> What is the impedance of the filter at the crossover frequency in a well-designed loop?</summary>

Approximately $R_1$ (the flat region between the zero and the third pole), slightly reduced by $C_1/(C_1+C_2)$.

</details>

<details><summary><b>Q5.</b> Where should the zero and the third pole sit relative to crossover?</summary>

Symmetrically on a log scale: $\omega_z = \omega_c/b$, $\omega_{p3} = b\,\omega_c$, with $b = \sqrt{1+C_1/C_2}$; this maximises phase margin for a given capacitor ratio.

</details>

<details><summary><b>Q6.</b> Which component dominates the loop-filter area and why?</summary>

C1. It sets the zero at low frequency (below crossover) with a resistor that must stay small for noise: $C_1 = 1/(\omega_z R_1)$.

</details>

<details><summary><b>Q7.</b> How does the resistor's thermal noise reach the output?</summary>

As a voltage $\sqrt{4kTR_1}$ on $V_{cont}$ (shaped by the capacitors), turned into frequency noise by $K_{VCO}$ and high-pass filtered by the loop like VCO noise. Larger R1 → more noise; FFNC can cancel it (Phase 6).

</details>

<details><summary><b>Q8.</b> What is an active loop filter and when is it used?</summary>

An op-amp integrator realising $Z(s)$; it allows a $V_{cont}$ range independent of the CP compliance. It adds op-amp noise and power; most integrated PLLs use passive filters.

</details>

<details><summary><b>Q9.</b> How do you implement 13 pF in SKY130?</summary>

MIM caps at about 2 fF/µm² (≈6500 µm² as 30×30 µm tiles) or a 1.8 V MOS cap at ~7.5 fF/µm² (≈1760 µm²), the latter only if biased above ~0.6 V (cheat sheet).

</details>

<details><summary><b>Q10.</b> What does the phase of Z do between the zero and the pole?</summary>

It rises from −90° toward 0° after the zero and falls back to −90° after the pole; the peak phase is $\arctan b - \arctan(1/b)$ at the geometric mean frequency.

</details>

<details><summary><b>Q11.</b> What is the effect of R1 C1 process spread on the loop?</summary>

The zero moves (±30% across R/C corners in SKY130, step 7 finding); phase margin and peaking change. Designs centre PM so that extreme R/C corners still keep about 45°.

</details>

<a id='step-16'></a>
## Step 16 — Divider: why ÷N divides both frequency and phase

*Roadmap step 16: waveform sketch*

### What happens
A ÷N counter produces one output cycle for every $N$ input cycles. Its output frequency is $f_{in}/N$, and a phase
change at its input appears **divided by N** at its output.

### How
If the input edges move by a time $\Delta t$, the output edges (which are copies of every $N$-th input edge) move by the
same $\Delta t$. In radians: the input phase change is $\Delta\phi_{in} = 2\pi f_{in}\Delta t$ and the output phase change is
$\Delta\phi_{out} = 2\pi \frac{f_{in}}{N}\Delta t = \Delta\phi_{in}/N$. So in the phase domain
$$\Phi_{div}(s) = \frac{1}{N}\Phi_{out}(s).$$

### Why it matters
In the loop, the divider is the $1/N$ in the feedback path. Its consequences: (1) the closed loop multiplies the input
phase (and anything added at the PD input) by $N$ at low frequency, i.e. **noise power ×N²**; (2) the divider is also a
*sampler* — it only passes information once per output period; (3) time jitter is preserved, phase jitter is divided.

In [ ]:
fin, N = 1.0, 4                                  # normalised: input period 1, divide by 4 for readability
t = np.linspace(0, 12, 3000)
def frame(shift):
    tin = t - np.where(t > 4, shift, 0)          # input edges move by 'shift' after t = 4
    vin = (np.sin(2*np.pi*fin*tin) > 0).astype(float)
    vout = (np.sin(2*np.pi*fin*tin/N) > 0).astype(float)
    ref = (np.sin(2*np.pi*fin*t/N) > 0).astype(float)
    return ([dict(x=t, y=vin+2.4, name="input (period 1)"),
             dict(x=t, y=vout+1.2, name="divided output (/4)"),
             dict(x=t, y=ref, name="divided output without shift", line=dict(dash="dot"))],
            f"time shift {shift:.2f}: input phase {360*shift:.0f} deg -> output phase {360*shift/N:.0f} deg (same time shift)")
fig = sweep([0.0, 0.1, 0.25, 0.5, 1.0], frame, "input time shift (input periods)", active=2, height=420,
            layout=lambda f: f.update_xaxes(title="time (input periods)").update_yaxes(showticklabels=False))
show(fig)

### Interview questions — step 16 (10)

<details><summary><b>Q1.</b> Why does a divider divide phase by N?</summary>

It preserves time shifts of edges, but the output period is N times longer, so the same time shift is a phase N times smaller.

</details>

<details><summary><b>Q2.</b> Does a divider reduce jitter?</summary>

It preserves absolute time jitter of the edges it passes (plus its own), but phase noise in dBc/Hz drops by 20log N because the frequency is N times lower.

</details>

<details><summary><b>Q3.</b> What is a TSPC divider and why is it used at GHz?</summary>

True single-phase clocked flip-flop: dynamic logic with one clock phase, few transistors and small load; fast enough for the first ÷2 stages at GHz (our ÷32 starts with TSPC stages).

</details>

<details><summary><b>Q4.</b> Why is divider noise multiplied by N² at the PLL output?</summary>

Divider jitter appears at the PD input like reference jitter; the loop forces the output phase to N times the PD input phase inside the bandwidth, so noise power is ×N².

</details>

<details><summary><b>Q5.</b> How does a sub-sampling PLL avoid the divider in the loop?</summary>

The SSPD samples the VCO directly with the reference; the loop compares the VCO phase with N times the reference phase without dividing. A divider is only used in the FLL for acquisition and turned quiet once locked.

</details>

<details><summary><b>Q6.</b> What is the difference between a synchronous and an asynchronous (ripple) divider?</summary>

Synchronous: all flip-flops clocked by the input, output edge timed by the input clock (low jitter). Ripple: each stage clocks the next, delays and jitter accumulate; fine for slow taps, bad for the feedback path.

</details>

<details><summary><b>Q7.</b> Why do dividers need a retiming flip-flop in low-jitter PLLs?</summary>

To resynchronise the divided edge to the clean VCO edge, removing the accumulated jitter of the counter.

</details>

<details><summary><b>Q8.</b> What is a dual-modulus prescaler?</summary>

A divider that switches between N and N+1 under control; combined with counters it gives programmable (and fractional, with a modulator) division.

</details>

<details><summary><b>Q9.</b> Is a divider linear in the phase domain?</summary>

Yes for small phase changes ($\Phi_{div} = \Phi_{out}/N$), but it is a sampled system; its output carries information only at its edges.

</details>

<details><summary><b>Q10.</b> What is the power of a ÷32 TSPC chain at 1.6 GHz roughly?</summary>

Dominated by the first stage: $P \approx C V_{DD}^2 f$; with ~20 fF switched at 1.6 GHz and 1.8 V that is ~0.1 mW per stage, halving each stage; the proposal budgets 0.7 mW for the whole FLL.

</details>

<a id='step-17'></a>
## Step 17 — Linear phase-domain model: G(s), H(s), E(s)

*Roadmap step 17: block diagram and equations*

### What happens
Once locked, all signals in the loop are small deviations of phase. Each block becomes a transfer function and the
PLL becomes a linear feedback system.

### How
Forward path: PD gain $K_{PD}$, filter $F(s)$ (or $Z(s)$ for a CP), VCO $K_{VCO}/s$; feedback path $1/N$.
The **open-loop gain** (loop gain) is
$$G(s) = \frac{K_{PD}\,F(s)\,K_{VCO}}{N\,s}\qquad\text{(CP-PLL: } G(s) = \frac{I_p}{2\pi}Z(s)\frac{K_{VCO}}{Ns}).$$
The closed-loop transfer from input phase to output phase and the error transfer are
$$\boxed{H(s) = \frac{\Phi_{out}}{\Phi_{in}} = N\frac{G(s)}{1+G(s)}},\qquad
\frac{\Phi_e}{\Phi_{in}} = E(s) = \frac{1}{1+G(s)}.$$
(Razavi writes $H(s)$ for $N=1$; here the factor $N$ is kept explicit.)

### Why
$|G|\gg1$ at low frequency → $H\to N$: the output follows $N\times$ the input; the error vanishes. $|G|\ll1$ at high
frequency → $H\to0$: the output ignores the input and keeps the VCO's own phase. The **crossover** $|G(j\omega_c)|=1$
is the loop bandwidth. The same $1/(1+G)$ shapes VCO noise (step 23).

In [ ]:
import os, sys
f = np.logspace(3, 9, 500)
Ip, Kv, N = 100e-6, 2*np.pi*300e6, 32
def frame(fc_MHz):
    d = sl.loop_design_type2(fc_MHz*1e6, 60, Ip, Kv, N)
    Z = lambda s: (1+s*d['R']*d['C1'])/(s*(d['C1']+d['C2'])*(1+s*d['R']*d['C1']*d['C2']/(d['C1']+d['C2'])))
    G = lambda s: Ip/(2*np.pi)*Z(s)*Kv/(N*s)
    s = 2j*np.pi*f
    g = G(s)
    return ([dict(x=f, y=20*np.log10(abs(g)), name="|G| open loop"),
             dict(x=f, y=20*np.log10(abs(g/(1+g))), name="|H/N| closed loop"),
             dict(x=f, y=20*np.log10(abs(1/(1+g))), name="|E| = |1/(1+G)|")],
            f"f_c = {fc_MHz} MHz, PM 60 deg: R1 = {d['R']/1e3:.1f} kOhm, C1 = {d['C1']*1e12:.1f} pF, C2 = {d['C2']*1e12:.2f} pF")
fig = sweep([0.5, 1.0, 1.5, 2.0, 2.5, 3.5, 5.0], frame, "loop bandwidth f_c (MHz)", active=4,
            layout=lambda f: f.update_xaxes(type="log", title="offset frequency (Hz)")
                              .update_yaxes(title="dB", range=[-60, 80]))
show(fig)

# cross-check with ngspice (learn/spice/pll_phase_domain.cir, f_c = 2.5 MHz design)
a = np.loadtxt("data/pll_phase_domain.csv")
d = sl.loop_design_type2(2.5e6, 60, Ip, Kv, N)
Z = lambda s: (1+s*d['R']*d['C1'])/(s*(d['C1']+d['C2'])*(1+s*d['R']*d['C1']*d['C2']/(d['C1']+d['C2'])))
G = lambda s: Ip/(2*np.pi)*Z(s)*Kv/(N*s)
g = G(2j*np.pi*a[:, 0])
print("ngspice vs Python, max |H| difference: %.2e dB" % np.max(abs(a[:, 1] - 20*np.log10(abs(N*g/(1+g))))))

**Try it.** As $f_c$ grows, $|H/N|$ stays flat to a higher frequency and $|E|$ (the VCO-noise high-pass) suppresses
more. The filter values update: a wider loop needs a smaller $C_1$ (area ∝ $1/f_c^2$). The ngspice phase-domain netlist
(`learn/spice/pll_phase_domain.cir`) reproduces the Python $H(s)$ to $10^{-4}$ dB — the same model in a circuit simulator.

### Interview questions — step 17 (12)

<details><summary><b>Q1.</b> Write the open-loop gain of a charge-pump PLL.</summary>

$G(s) = \frac{I_p}{2\pi}\,Z(s)\,\frac{K_{VCO}}{s}\,\frac1N$.

</details>

<details><summary><b>Q2.</b> What is the low-frequency value of $H(s) = \Phi_{out}/\Phi_{in}$ and why?</summary>

N. When $|G|\to\infty$, $\Phi_{out}/N = \Phi_{in}$: the divided output tracks the input exactly.

</details>

<details><summary><b>Q3.</b> Define loop bandwidth.</summary>

Usually the crossover frequency where $|G| = 1$ (0 dB). The −3 dB bandwidth of $H$ is somewhat higher (by ~1.2–1.5× depending on PM).

</details>

<details><summary><b>Q4.</b> Why is $E(s) = 1/(1+G)$ important besides steady-state error?</summary>

It is also the transfer function from VCO phase noise to output: the loop high-pass-filters VCO noise with $1/(1+G)$.

</details>

<details><summary><b>Q5.</b> What assumptions does the linear phase model make?</summary>

Locked loop, small phase errors (inside the PD linear range), continuous-time averaging of sampled blocks ($f_c\ll f_{REF}$), linear $K_{VCO}$.

</details>

<details><summary><b>Q6.</b> How can you simulate a PLL's H(s) in SPICE without the GHz signals?</summary>

Build the phase-domain model with controlled sources: node voltages represent phases, a VCCS for the PD/CP, real R/C for the filter, a VCCS into a 1 F capacitor for the VCO integrator, a VCVS of 1/N for the divider; run an AC analysis.

</details>

<details><summary><b>Q7.</b> What does $|H|$ peaking indicate?</summary>

Low phase margin. Peaking amplifies reference and PD noise around the bandwidth and causes ringing in the step response.

</details>

<details><summary><b>Q8.</b> Name the poles and zeros of G(s) for a third-order CP-PLL.</summary>

Two poles at the origin (CP integrator, VCO), one zero at $1/R_1C_1$, one pole at $(C_1+C_2)/(R_1C_1C_2)$.

</details>

<details><summary><b>Q9.</b> Why does $C_1$ scale as $1/f_c^2$?</summary>

From the crossover condition $C_t = b I_pK_{VCO}/(2\pi N\omega_c^2)$: halving $f_c$ quadruples the total capacitance.

</details>

<details><summary><b>Q10.</b> Why is the closed-loop transfer called low-pass for the input and high-pass for the VCO?</summary>

$H/N = G/(1+G)$ → 1 at low frequency, 0 at high; $1/(1+G)$ → 0 at low, 1 at high. Their sum is 1 for any loop (complementary).

</details>

<details><summary><b>Q11.</b> Can a PLL bandwidth exceed $f_{REF}/2$?</summary>

No meaningful sense: the PD only updates once per reference period (Nyquist). Practical CP-PLLs stay below ~$f_{REF}/10$, SSPLLs ~$f_{REF}/20$ for spur and phase reasons.

</details>

<details><summary><b>Q12.</b> How do you measure the loop bandwidth on silicon?</summary>

Modulate the reference phase/frequency and measure the output response vs modulation frequency, or read the phase-noise plot: the shoulder where VCO noise takes over approximates $f_c$.

</details>

<a id='step-18'></a>
## Step 18 — Order and type

*Roadmap step 18: type = number of integrators (poles at the origin) in G(s)*

### What happens
Two numbers classify every PLL. **Type** = number of poles of the open-loop gain $G(s)$ at the origin
(integrators). **Order** = degree of the closed-loop denominator (number of closed-loop poles).

### How to read them off
| Loop | $G(s)$ | Type | Order |
|---|---|---|---|
| XOR/mixer + RC + VCO | $\frac{K_{PD}K_{VCO}}{N s(1+s/\omega_{LPF})}$ | I | 2 |
| PFD/CP + C1 + VCO | $\frac{I_pK_{VCO}}{2\pi N C_1 s^2}$ | II | 2 (unstable) |
| PFD/CP + R1C1 + VCO | $\frac{I_pK_{VCO}(1+sR_1C_1)}{2\pi N C_1 s^2}$ | II | 2 |
| PFD/CP + R1C1‖C2 + VCO | $\frac{I_pK_{VCO}(1+sR_1C_1)}{2\pi N (C_1+C_2)s^2(1+sR_1C_s)}$ | II | 3 |
| our SSPLL (pulsed $G_m$ + R1C1‖C2 + VCO, plus hold) | same shape, no $N$, extra hold delay | II | 3 (+ delay) |

### Why it matters
**Type sets the steady-state error** (step 20): Type I tracks a phase step with zero error but a frequency step with a
constant phase error; Type II tracks both with zero error. **Order sets the difficulty of stability**: each extra pole
costs phase. Higher type than II is rare in integrated PLLs because three integrators give −270° and are hard to stabilise.

In [ ]:
f = np.logspace(3, 9, 400); s = 2j*np.pi*f
G1 = 2*np.pi*2.5e6/s                                    # Type I, crossover 2.5 MHz
wz = 2*np.pi*2.5e6/3.73; wp = 2*np.pi*2.5e6*3.73
G2 = (2*np.pi*2.5e6)**2/3.73*(1+s/wz)/(s**2*(1+s/wp))    # Type II, crossover 2.5 MHz, PM 60
fig = make_subplots(rows=1, cols=2, subplot_titles=("|G| (dB)", "phase of G (deg)"))
for g, n in ((G1, "Type I: -20 dB/dec"), (G2, "Type II: -40 dB/dec, zero, pole")):
    fig.add_trace(go.Scatter(x=f, y=20*np.log10(abs(g)), name=n), 1, 1)
    fig.add_trace(go.Scatter(x=f, y=np.degrees(np.unwrap(np.angle(g))), name=n+" phase", showlegend=False), 1, 2)
fig.update_xaxes(type="log", title_text="frequency (Hz)")
show(fig, 420)

### Interview questions — step 18 (10)

<details><summary><b>Q1.</b> Define the type and the order of a PLL.</summary>

Type: number of open-loop poles at s = 0. Order: number of closed-loop poles (degree of $1+G$'s numerator polynomial).

</details>

<details><summary><b>Q2.</b> What type is a charge-pump PLL and why?</summary>

Type II: the CP into a capacitor and the VCO are both integrators.

</details>

<details><summary><b>Q3.</b> Why are Type-III PLLs rare?</summary>

Three integrators give −270°; two zeros are needed to recover phase margin, the loop becomes conditionally stable (unstable if the gain drops, e.g. during saturation) and acquisition is slow.

</details>

<details><summary><b>Q4.</b> What is the low-frequency slope of |G| for Type I and Type II?</summary>

−20 dB/decade and −40 dB/decade respectively.

</details>

<details><summary><b>Q5.</b> Is our SSPLL Type I or Type II?</summary>

Type II: the pulsed-Gm charge pump drives a capacitor (integrator) and the VCO integrates. Nagam & Kinget and Gao use the same structure.

</details>

<details><summary><b>Q6.</b> Is a second-order Type-II loop always stable?</summary>

With a zero (series R) yes for any gain in the CT model (root locus stays in the left half plane); without the zero it is marginally stable. Sampling effects still limit gain (Gardner).

</details>

<details><summary><b>Q7.</b> What does the third pole add, and is the loop still Type II?</summary>

It filters ripple and costs phase; the type is unchanged (still two poles at the origin); the order becomes three.

</details>

<details><summary><b>Q8.</b> Can a PLL have zero steady-state phase error for a frequency ramp?</summary>

Only if it is Type III. A Type-II loop tracks a frequency ramp with a constant phase error (step 20).

</details>

<details><summary><b>Q9.</b> How do you identify type from a step response measurement?</summary>

Apply a frequency step to the reference: a residual constant phase error → Type I; zero final error → Type II.

</details>

<details><summary><b>Q10.</b> What is a 'loop order' versus 'filter order'?</summary>

Filter order counts the filter's poles (e.g. C1 and C2 → one pole at zero plus one real pole); loop order adds the VCO integrator. A '2nd-order filter' gives a 3rd-order loop.

</details>

<a id='step-19'></a>
## Step 19 — Stability: Bode, crossover, phase margin, root locus

*Roadmap step 19: why 45–65° is chosen*

### What happens
A feedback loop becomes unstable if, at the frequency where $|G| = 1$, the phase reaches −180°: the fed-back signal
returns inverted twice (Barkhausen) and grows. **Phase margin** measures the distance from that point:
$$PM = 180° + \angle G(j\omega_c),\qquad |G(j\omega_c)| = 1.$$

### How, for the 3rd-order Type-II loop
$$\angle G(j\omega) = -180° + \arctan\frac{\omega}{\omega_z} - \arctan\frac{\omega}{\omega_{p3}}.$$
Placing $\omega_z = \omega_c/b$ and $\omega_{p3} = b\,\omega_c$ (symmetric on a log axis) maximises the phase at $\omega_c$:
$$PM_{max} = \arctan b - \arctan\frac1b,\qquad b^2 = \frac{\omega_{p3}}{\omega_z} = 1+\frac{C_1}{C_2}.$$
So $PM = 60°$ needs $b = \tan(45°+PM/2) = 3.73$, $C_1/C_2 = b^2-1 = 12.9$.

### Why 45–65°
Low PM → closed-loop peaking (noise amplification near $f_c$) and ringing/long settling. High PM → the zero and pole are far
apart: a huge $C_1$ (area) and weak ripple filtering, and for the same $f_c$ a slower step response. 50–60° is the usual
optimum for integrated jitter. The root locus shows the same: as loop gain varies, closed-loop poles move; good PM keeps
the complex pair well damped.

In [ ]:
from scipy import signal
fc = 2.5e6; wc = 2*np.pi*fc
f = np.logspace(4, 9, 500); s = 2j*np.pi*f
t = np.linspace(0, 3e-6, 1500)
def frame(pm):
    b_ = np.tan(np.radians(45+pm/2)); wz, wp = wc/b_, wc*b_; K = wc**2/b_
    g = K*(1+s/wz)/(s**2*(1+s/wp))
    sysH = signal.lti([K/wz, K], [1/wp, 1, K/wz, K])        # H/N = G/(1+G)
    _, y = signal.step(sysH, T=t)
    peak = 20*np.log10(np.max(abs(g/(1+g))))
    return ([dict(x=f, y=20*np.log10(abs(g)), name="|G| (dB)", row=1, col=1),
             dict(x=f, y=np.degrees(np.unwrap(np.angle(g))), name="phase G (deg)", row=1, col=2),
             dict(x=f, y=20*np.log10(abs(g/(1+g))), name="|H/N| (dB)", row=1, col=1),
             dict(x=t*1e6, y=y, name="phase-step response", row=2, col=1),
             dict(x=t*1e6, y=np.ones_like(t), name="final value", line=dict(dash="dot"), row=2, col=1)],
            f"PM {pm} deg: C1/C2 = {b_**2-1:.1f}, closed-loop peaking {peak:.2f} dB, overshoot {100*(y.max()-1):.0f} %")
fig = sweep([30, 40, 45, 50, 55, 60, 65, 70, 75], frame, "phase margin (deg)", active=5, rows=2, cols=2,
            subplot_titles=("magnitude", "phase of G", "step response of H/N", ""), height=640,
            layout=lambda f: f.update_xaxes(type="log", row=1, col=1).update_xaxes(type="log", row=1, col=2)
                              .update_yaxes(range=[-40, 60], row=1, col=1).update_yaxes(range=[-200, -90], row=1, col=2)
                              .update_xaxes(title_text="time (us)", row=2, col=1))
show(fig)

**Root locus.** Keep the zero and pole of the 60° design fixed and scale the loop gain (e.g. $I_p$ or $K_{VCO}$ changing over PVT). The closed-loop poles move; the title shows the phase margin you would actually get.

In [ ]:
b_ = np.tan(np.radians(75)); wz, wp = wc/b_, wc*b_; K0 = wc**2/b_
def frame(m):
    K = K0*m
    p = np.roots([1/wp, 1, K/wz, K])
    g = K*(1+s/wz)/(s**2*(1+s/wp)); fcx = sl.crossover(f, 20*np.log10(abs(g)))
    pm = 180 + np.degrees(np.angle(K*(1+2j*np.pi*fcx/wz)/((2j*np.pi*fcx)**2*(1+2j*np.pi*fcx/wp))))
    locus = [np.roots([1/wp, 1, K0*mm/wz, K0*mm]) for mm in np.logspace(-1.5, 1.5, 200)]
    L = np.array(locus)
    return ([dict(x=L.real.flatten()/wc, y=L.imag.flatten()/wc, mode="markers", marker=dict(size=3, color="#cbd5e1"), name="locus"),
             dict(x=p.real/wc, y=p.imag/wc, mode="markers", marker=dict(size=12), name="closed-loop poles"),
             dict(x=[-1/b_], y=[0], mode="markers", marker=dict(symbol="circle-open", size=12), name="zero")],
            f"gain x{m}: crossover {fcx/1e6:.2f} MHz, PM {pm:.0f} deg")
fig = sweep([0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0], frame, "loop-gain multiplier", active=3,
            layout=lambda f: f.update_xaxes(title="Re{s}/w_c", range=[-5, 0.5]).update_yaxes(title="Im{s}/w_c", range=[-3, 3]))
show(fig, 480)

**Try it.** Drop the PM to 30°: 6 dB of peaking and a 52% overshoot. At 60°: 1.7 dB and 19%. At 75°: 0.8 dB but
$C_1/C_2 = 57$ (huge $C_1$). In the root locus, a 4× gain change moves the crossover by ~3× and costs ~15° of margin —
why designs centre PM near 60° when $K_{VCO}$ and the CP gain spread with PVT.

### Interview questions — step 19 (13)

<details><summary><b>Q1.</b> Define phase margin.</summary>

$PM = 180° + \angle G(j\omega_c)$ where $|G(j\omega_c)| = 1$. It is how much additional phase lag would make the loop oscillate.

</details>

<details><summary><b>Q2.</b> Derive the maximum phase margin of a 3rd-order Type-II loop.</summary>

$\angle G = -180° + \arctan(\omega/\omega_z) - \arctan(\omega/\omega_p)$; maximised at $\omega=\sqrt{\omega_z\omega_p}$; with $b=\sqrt{\omega_p/\omega_z}$, $PM_{max} = \arctan b - \arctan(1/b)$.

</details>

<details><summary><b>Q3.</b> Relate $C_1/C_2$ to the pole/zero spacing.</summary>

$\omega_p/\omega_z = (C_1+C_2)/C_2 = 1 + C_1/C_2 = b^2$.

</details>

<details><summary><b>Q4.</b> What $C_1/C_2$ gives 60° and 45° phase margin?</summary>

60°: $b = \tan 75° = 3.73$, $C_1/C_2 = 12.9$. 45°: $b = \tan 67.5° = 2.41$, $C_1/C_2 = 4.8$.

</details>

<details><summary><b>Q5.</b> Why not choose 80° phase margin?</summary>

It needs $C_1/C_2 \approx 130$ (enormous $C_1$), gives poor ripple filtering, and slows the transient for the same crossover.

</details>

<details><summary><b>Q6.</b> Relationship between PM and closed-loop peaking?</summary>

For the 3rd-order Type-II loop with symmetric zero/pole: 60° → 1.7 dB, 45° → ~3 dB, 30° → ~6 dB (see the slider). Peaking multiplies in-band noise near $f_c$, so lower PM raises integrated jitter.

</details>

<details><summary><b>Q7.</b> What is gain margin and why is it rarely quoted for PLLs?</summary>

How much the gain can grow before instability. A Type-II CT loop has no −180° crossing above $\omega_c$ (phase returns to −180° only asymptotically), so gain margin is infinite in the CT model; the real limit comes from sampling delay (Gardner).

</details>

<details><summary><b>Q8.</b> What happens to PM if $K_{VCO}$ doubles?</summary>

Crossover moves up toward the third pole; PM falls (and $f_c$ rises ~1.6–2×). If $K_{VCO}$ halves, crossover moves toward the zero; PM also falls. Maximum PM only at the design gain.

</details>

<details><summary><b>Q9.</b> What is conditional stability?</summary>

A loop stable at the design gain but unstable if the gain *decreases* (phase dips below −180° at lower frequencies). Type-III loops and loops with extra low-frequency poles can be conditionally stable.

</details>

<details><summary><b>Q10.</b> How does the root locus of a Type-II loop with a zero look?</summary>

Two poles start at the origin, move into the left half plane on a circle around the zero and return to the real axis; with the third pole, one branch goes to −∞ and the pair can approach the imaginary axis at very high gain.

</details>

<details><summary><b>Q11.</b> What is the relation between ζ and PM for second-order loops?</summary>

Approximately $PM \approx 100\zeta$ degrees for $\zeta < 0.7$ (e.g. ζ = 0.5 → ~51°).

</details>

<details><summary><b>Q12.</b> How does a pure time delay $T_d$ affect PM?</summary>

It adds phase $-\omega_c T_d$ (rad) without changing magnitude: PM loss $= 360° f_c T_d$. A half-period hold ($T_d = T_{REF}/2$) at $f_c = f_{REF}/10$ costs 18°.

</details>

<details><summary><b>Q13.</b> How would you measure phase margin in simulation?</summary>

Break the loop in the phase-domain model (or with a behavioural model) and run AC; or fit the closed-loop step response overshoot; in transistor-level sims, extract the block gains and compute G(s).

</details>

<a id='step-20'></a>
## Step 20 — Time response and steady-state error

*Roadmap step 20: phase step, frequency step, ramp; final-value theorem per type*

### What happens
Apply three standard inputs to the reference phase: a **phase step** $\Phi_{in}=\Delta\phi/s$, a **frequency step**
$\Phi_{in}=\Delta\omega/s^2$ (phase ramp), and a **frequency ramp** $\Phi_{in}=\dot\omega/s^3$. Watch the phase error.

### How: the final-value theorem
$$\phi_{e}(\infty) = \lim_{s\to0} s\,E(s)\,\Phi_{in}(s),\qquad E = \frac1{1+G}.$$
| Input | Type I ($G\approx K/s$) | Type II ($G\approx K/s^2$) |
|---|---|---|
| phase step $\Delta\phi/s$ | 0 | 0 |
| frequency step $\Delta\omega/s^2$ | $\Delta\omega/K$ | 0 |
| frequency ramp $\dot\omega/s^3$ | ∞ | $\dot\omega/K$ |

### Why
Each integrator in the loop "absorbs" one order of input: to hold a new frequency the VCO needs a new $V_{cont}$. In
Type I that voltage can only come from a **standing phase error** through $K_{PD}$; in Type II the filter capacitor
stores it, so the error can return to zero.

In [ ]:
from scipy import signal
wc = 2*np.pi*2.5e6
t = np.linspace(0, 4e-6, 2000)
E1 = signal.lti([1, 0], [1, wc])                                     # Type I: G = wc/s
wz = wc/3.73; K2 = wc**2/3.73*1.0
E2 = signal.lti([1, 0, 0], [1, K2/wz, K2])                           # Type II, 2nd order: G = K(1+s/wz)/s^2
inputs = {"phase step (1 rad)": np.ones_like(t),
          "frequency step (1 Mrad/s)": 1e6*t,
          "frequency ramp (1e12 rad/s^2)": 0.5e12*t**2}
def frame(name):
    u = inputs[name]
    _, e1, _ = signal.lsim(E1, u, t); _, e2, _ = signal.lsim(E2, u, t)
    return ([dict(x=t*1e6, y=u, name="input phase (rad)", line=dict(dash="dot")),
             dict(x=t*1e6, y=e1, name="error, Type I"), dict(x=t*1e6, y=e2, name="error, Type II")],
            f"{name}: final error Type I {e1[-1]:.3g} rad, Type II {e2[-1]:.3g} rad")
fig = sweep(list(inputs), frame, "input", fmt="{}", active=1,
            layout=lambda f: f.update_xaxes(title="time (us)").update_yaxes(title="rad", range=[-0.3, 1.3]))
show(fig, 450)

**Try it.** Frequency step: Type I settles at $\Delta\omega/K = 10^6/(2\pi\cdot2.5\cdot10^6) = 0.064$ rad, Type II returns to 0.
Frequency ramp: Type I grows without bound, Type II settles at $\dot\omega/K = 10^{12}/K_2$.

### Interview questions — step 20 (10)

<details><summary><b>Q1.</b> State the final-value theorem.</summary>

$\lim_{t\to\infty} x(t) = \lim_{s\to0} sX(s)$, valid if all poles of $sX(s)$ are in the left half plane.

</details>

<details><summary><b>Q2.</b> Steady-state phase error of a Type-I loop for a frequency step $\Delta\omega$?</summary>

$\Delta\omega/K$ with $K = K_{PD}K_{VCO}/N$ (the DC loop gain constant).

</details>

<details><summary><b>Q3.</b> Why can a Type-II loop track a frequency step with zero phase error?</summary>

The integrating filter holds the new control voltage on its capacitor; no phase error is needed to sustain it.

</details>

<details><summary><b>Q4.</b> Steady-state error of a Type-II loop for a frequency ramp?</summary>

Constant: $\dot\omega/K$ where $G\approx K/s^2$ at low frequency ($K = I_pK_{VCO}/(2\pi N(C_1+C_2))$).

</details>

<details><summary><b>Q5.</b> Where does a frequency ramp occur in practice?</summary>

Temperature drift of the reference, Doppler in receivers, spread-spectrum clocking (triangular frequency modulation).

</details>

<details><summary><b>Q6.</b> How long does a Type-II PLL take to settle after a phase step?</summary>

Roughly $4/(\zeta\omega_n)$ for 2% settling; with $f_c = 2.5$ MHz and PM ≈ 60°, of order 0.5–1 µs. Lock time from a large frequency step is dominated by frequency acquisition, not by this.

</details>

<details><summary><b>Q7.</b> Why is the proposal's lock time spec 5 µs typ / 20 µs max, much longer than 1/f_c?</summary>

It includes frequency acquisition by the FLL (slewing the VCO across its offset) and hand-over to the SSPD loop, then linear settling to ±10 ps.

</details>

<details><summary><b>Q8.</b> What is cycle slipping in a step response?</summary>

If the transient phase error exceeds the PD's linear range (±π for an SSPD, ±2π for a PFD), the error wraps and the loop loses a cycle; the linear step response no longer applies.

</details>

<details><summary><b>Q9.</b> What is the error response to a phase step in Type I first-order loop?</summary>

$\phi_e(t) = \Delta\phi\,e^{-Kt}$: a single exponential with time constant $1/K$.

</details>

<details><summary><b>Q10.</b> Why does overshoot in a Type-II step response happen even with high PM?</summary>

The zero in $H(s)$: the closed-loop transfer has a zero at $\omega_z$ which adds overshoot independent of damping (the step response overshoots even for ζ ≥ 1).

</details>

<a id='step-21'></a>
## Step 21 — Acquisition: lock range, pull-in range, capture range, cycle slips

*Roadmap step 21: explanation and simulation*

### What happens
Linear analysis assumes lock. Starting from a frequency difference $\Delta\omega$, a loop with a sinusoidal (mixer-like)
PD may lock quickly, lock after slipping cycles, or never lock.

### How: first-order loop with a sinusoidal PD
$$\frac{d\phi_e}{dt} = \Delta\omega - K\sin\phi_e .$$
- If $|\Delta\omega| \le K$, there is an equilibrium $\sin\phi_e = \Delta\omega/K$: **lock** (hold-in range $= K$).
- If $|\Delta\omega| > K$, $d\phi_e/dt$ never reaches 0: the phase error keeps growing — **cycle slips** — at an average
  beat frequency $\omega_{beat} = \sqrt{\Delta\omega^2-K^2}$.

Definitions (Gardner/Best): **hold-in (lock) range**: static range over which lock can be maintained;
**lock-in (capture) range**: range over which the loop locks without slipping a cycle (≈ loop bandwidth);
**pull-in range**: range over which it eventually locks after slipping cycles (in Type II with a sinusoidal PD the filter
integrator slowly pulls it in); **pull-out**: largest frequency step that does not cause a slip.

### Why a PFD solves it
A PFD's average output keeps the right sign whatever the frequency difference, so a CP-PLL pulls in over the entire VCO
range. An SSPD behaves like a sinusoidal PD with period $T_{VCO}$ (it cannot tell harmonics apart) — the reason our PLL
needs an FLL (Phase 4, step 52).

In [ ]:
from scipy.integrate import solve_ivp
K = 2*np.pi*1e6                                  # loop constant K = 2*pi*1 MHz: hold-in range 1 MHz
t = np.linspace(0, 10e-6, 3000)
def frame(r):
    dw = r*K
    sol = solve_ivp(lambda tt, p: [dw - K*np.sin(p[0])], (0, t[-1]), [0.0], t_eval=t, max_step=2e-9)
    pe = sol.y[0]
    ferr = (dw - K*np.sin(pe))/(2*np.pi)
    msg = "locks (phase error settles)" if r <= 1 else f"no lock: beat {np.sqrt(dw**2-K**2)/2/np.pi/1e3:.0f} kHz"
    return ([dict(x=t*1e6, y=pe/(2*np.pi), name="phase error (cycles)", row=1, col=1),
             dict(x=t*1e6, y=ferr/1e3, name="frequency error (kHz)", row=2, col=1)],
            f"dw/K = {r}: {msg}")
fig = sweep([0.3, 0.6, 0.9, 0.99, 1.01, 1.1, 1.5, 2.0], frame, "frequency offset dw / K", active=4, rows=2, cols=1,
            shared_xaxes=True, subplot_titles=("phase error (cycles)", "frequency error (kHz)"), height=560,
            layout=lambda f: f.update_xaxes(title_text="time (us)", row=2, col=1))
show(fig)

**Try it.** Just above $\Delta\omega/K = 1$ the phase error creeps, then slips a whole cycle quickly, again and again: the frequency error is a pulse train, its average is the beat frequency. Below 1 it settles at $\arcsin(\Delta\omega/K)$.

### Interview questions — step 21 (10)

<details><summary><b>Q1.</b> Define hold-in, pull-in, lock-in (capture) and pull-out ranges.</summary>

Hold-in: frequency range over which lock is maintained statically. Lock-in: offset range from which the loop locks within one beat (no cycle slip). Pull-in: range from which it eventually locks, possibly after slips. Pull-out: largest frequency step that does not cause a slip.

</details>

<details><summary><b>Q2.</b> What is the hold-in range of a first-order loop with a sinusoidal PD?</summary>

$|\Delta\omega| \le K$ where $K = K_{PD}K_{VCO}$ (the loop constant).

</details>

<details><summary><b>Q3.</b> What is the beat frequency when the loop is out of lock?</summary>

$\sqrt{\Delta\omega^2 - K^2}$ (Adler-like equation).

</details>

<details><summary><b>Q4.</b> Why does a PFD-based PLL have a pull-in range equal to the VCO range?</summary>

The PFD produces a non-zero average with the correct sign for any frequency difference, so the CP keeps pushing the VCO toward the reference.

</details>

<details><summary><b>Q5.</b> Why can a sub-sampling PD lock to the wrong frequency?</summary>

It samples at $f_{REF}$ and cannot distinguish $f_{VCO} = Nf_{REF}$ from $(N\pm1)f_{REF}$: all give a DC sample. An FLL or a PFD-based acquisition loop must first set the right harmonic.

</details>

<details><summary><b>Q6.</b> What is a cycle slip and why is it bad?</summary>

The phase error passes ±π (or ±2π) and the loop locks one cycle away. In data links it causes bit errors; in frequency synthesis, momentary frequency error; in our FFNC it means a large transient.

</details>

<details><summary><b>Q7.</b> How does loop bandwidth relate to the lock-in range?</summary>

The lock-in (capture) range is of the order of the loop bandwidth: offsets inside it are captured without slipping.

</details>

<details><summary><b>Q8.</b> What is false lock in a mixer PLL?</summary>

Locking to a harmonic or sideband (e.g. with square-wave inputs locking at $f_{in}\cdot m/n$), producing a wrong output frequency with the PD output at its lock value.

</details>

<details><summary><b>Q9.</b> How do you speed up acquisition?</summary>

Coarse frequency calibration (digital band selection), a frequency detector (PFD/FLL), temporarily widening the bandwidth (gear shifting), pre-charging $V_{cont}$.

</details>

<details><summary><b>Q10.</b> Why does our design use a dead-zone FLL instead of a plain PFD loop in parallel?</summary>

A PFD loop would keep injecting charge-pump noise and mismatch spurs after lock. The dead zone makes it quiet once the phase error is small, leaving the low-noise SSPD loop in control.

</details>

<a id='step-22'></a>
## Step 22 — Noise basics: L(f), ring-oscillator noise, jitter and σ_t from ∫L(f)

*Roadmap step 22: worked integration*

### What happens
A real oscillator's zero crossings wander. In frequency domain this is **phase noise**: power spread around the carrier.

### How it is defined and measured
For a signal $A\cos(\omega_0t + \phi_n(t))$ with small $\phi_n$:
$$\mathcal L(\Delta f) = \frac{\text{power in 1 Hz at } f_0+\Delta f}{\text{carrier power}}\ \ [\text{dBc/Hz}],\qquad
S_\phi(\Delta f) \approx 2\,\mathcal L(\Delta f)\ [\text{rad}^2/\text{Hz, one-sided}].$$
**Absolute (rms) jitter** over a band:
$$\boxed{\sigma_t = \frac{1}{2\pi f_0}\sqrt{\int_{f_1}^{f_2} 2\cdot10^{\mathcal L(f)/10}\,df}}$$
Other jitter definitions: **period jitter** (variation of one period), **cycle-to-cycle** (difference of adjacent periods).
For a free-running oscillator with white noise sources, timing error grows as a random walk, $\sigma_{\Delta T}=\kappa\sqrt{\Delta T}$, and
$$\mathcal L(\Delta f) = \frac{\sigma_{per}^2 f_0^3}{\Delta f^2}\qquad(\sigma_{per} = \text{period jitter}).$$
Check with our Spectre transient-noise result: $\sigma_{per} = 88$ fs at 1.602 GHz → $\mathcal L(1\,\text{MHz}) = -105$ dBc/Hz
(the thermal part; flicker adds on top).

### Why rings are noisy (Hajimiri & Abidi)
For a ring oscillator with power $P$ (white noise, all stages switching),
$$\mathcal L(\Delta f) \approx \frac{8}{3\eta}\,\frac{kT}{P}\,\frac{V_{DD}}{V_{char}}\,\frac{f_0^2}{\Delta f^2},\qquad \eta\approx 0.75\text{–}1,$$
independent of the number of stages for fixed $P$ and $f_0$: phase noise improves only with power (1/P). This gives a
practical ceiling of the figure of merit $FoM = -\mathcal L + 20\log(f_0/\Delta f) - 10\log(P/1\,\text{mW}) \approx 160$–$165$ dB.

In [ ]:
import sys; sys.path.insert(0, "../models")
import pllmodel as pm
j, f, L = pm.jitter(pm.TYP, "ffnc", band=(1e3, 200e6))       # proposal design point, FFNC on
def frame(f1):
    m = (f >= f1) & (f <= 40e6)
    cum = np.sqrt(np.cumsum(2*10**(L[m]/10) * np.gradient(f[m])))/(2*np.pi*pm.TYP.f0)*1e12
    return ([dict(x=f, y=L, name="L(f) (dBc/Hz)", row=1, col=1),
             dict(x=f[m], y=cum, name="cumulative jitter (ps)", row=2, col=1)],
            f"rms jitter from {f1/1e3:.0f} kHz to 40 MHz = {cum[-1]:.2f} ps (spec band 10 kHz-40 MHz)")
fig = sweep([1e3, 3e3, 1e4, 3e4, 1e5, 3e5, 1e6], frame, "lower integration limit (Hz)", fmt="{:.0e}", active=2,
            rows=2, cols=1, shared_xaxes=True, subplot_titles=("phase noise at 1.6 GHz, FFNC on", "jitter accumulated from f1 up to f"),
            height=600, layout=lambda f: f.update_xaxes(type="log").update_xaxes(title_text="offset (Hz)", row=2, col=1))
show(fig)

In [ ]:
# Ring-oscillator phase noise vs power for a given FoM (L(1 MHz) at 1.6 GHz), with our three estimates
Pm = np.logspace(-1, 1.5, 100)               # mW
fig = go.Figure()
for fom in (155, 160, 165):
    fig.add_trace(go.Scatter(x=Pm, y=-fom + 20*np.log10(1.6e9/1e6) - 10*np.log10(Pm), name=f"FoM {fom} dB"))
for name, Lv, P in (("ngspice ISF (proposal basis)", -97.35, 4.86), ("Spectre transient noise", -99.0, 4.86),
                    ("Spectre pnoise (optimistic)", -102.1, 4.86)):
    fig.add_trace(go.Scatter(x=[P], y=[Lv], mode="markers", marker=dict(size=11), name=name))
fig.update_xaxes(type="log", title="oscillator power incl. CCO (mW)").update_yaxes(title="L(1 MHz) at 1.6 GHz (dBc/Hz)")
show(fig, 430)

**Try it.** Move the lower limit from 10 kHz to 1 kHz: the jitter barely changes (little noise there), while the
cumulative curve rises steeply between ~0.5 and 5 MHz — the loop-bandwidth hump dominates, which is exactly the
band FFNC attacks. (Our ring estimates sit at FoM ≈ 155–159 dB on total CCO power, below the ~165 dB bound, because the
power counts the 3.3 V CCO mirror.)

### Interview questions — step 22 (15)

<details><summary><b>Q1.</b> Define L(Δf).</summary>

Single-sideband noise power in a 1 Hz bandwidth at offset Δf from the carrier, divided by the carrier power, in dBc/Hz.

</details>

<details><summary><b>Q2.</b> How are $S_\phi$ and L related?</summary>

For small phase deviations $S_\phi(f) \approx 2\mathcal L(f)$ (one-sided $S_\phi$), since the phase noise splits into two sidebands.

</details>

<details><summary><b>Q3.</b> How do you convert phase noise to rms jitter?</summary>

$\sigma_t = \sqrt{2\int 10^{\mathcal L/10}df}/(2\pi f_0)$ over the stated band.

</details>

<details><summary><b>Q4.</b> Quick estimate: flat -100 dBc/Hz from 0 to 1 MHz at 1.6 GHz, what jitter?</summary>

$\sigma_\phi = \sqrt{2\cdot10^{-10}\cdot10^6} = 0.0141$ rad; $\sigma_t = 0.0141/(2\pi\cdot1.6\times10^9) = 1.4$ ps.

</details>

<details><summary><b>Q5.</b> Why does free-running phase noise fall at 20 dB/decade?</summary>

White frequency noise integrates to a random walk of phase; $S_\phi \propto S_\omega/\omega^2$ → $1/f^2$. Flicker sources give $1/f^3$ near the carrier.

</details>

<details><summary><b>Q6.</b> What is the 1/f³ corner and why is it not equal to the device flicker corner?</summary>

The offset where flicker-induced phase noise equals white-induced. Up-conversion depends on the dc value of the ISF (waveform symmetry), so the 1/f³ corner can be well below or above the device 1/f corner.

</details>

<details><summary><b>Q7.</b> State Hajimiri's ring-oscillator phase-noise result and its main conclusion.</summary>

$\mathcal L \approx \frac{8}{3\eta}\frac{kT}{P}\frac{V_{DD}}{V_{char}}\frac{f_0^2}{\Delta f^2}$; for fixed power and frequency it does not depend on the number of stages; only more power (or less $V_{DD}/V_{char}$) helps.

</details>

<details><summary><b>Q8.</b> Define the oscillator FoM.</summary>

$FoM = -\mathcal L(\Delta f) + 20\log(f_0/\Delta f) - 10\log(P/1\,mW)$. Rings ≲ 165 dB, good LC oscillators 185–195 dB.

</details>

<details><summary><b>Q9.</b> Difference between period jitter, cycle-to-cycle jitter and absolute jitter?</summary>

Period: rms deviation of each period from the mean. Cycle-to-cycle: rms difference of adjacent periods. Absolute (TIE): rms deviation of edges from ideal positions — what integrated phase noise gives.

</details>

<details><summary><b>Q10.</b> Relate period jitter to phase noise for a free-running oscillator.</summary>

$\mathcal L(\Delta f) = \sigma_{per}^2 f_0^3/\Delta f^2$ (white noise region).

</details>

<details><summary><b>Q11.</b> Why does the absolute jitter of a free-running oscillator diverge?</summary>

Timing error accumulates as a random walk ($\sigma\propto\sqrt{t}$); a PLL bounds it by correcting the phase at every reference edge.

</details>

<details><summary><b>Q12.</b> Why do Spectre pnoise and transient noise disagree for our ring?</summary>

Sideband convention of the converted pnoise output (factor 2, 3 dB) — found in step 7; device noise itself matched to 0.0 dB.

</details>

<details><summary><b>Q13.</b> What is AM-to-PM conversion?</summary>

Amplitude noise converted to phase noise by nonlinear capacitances/thresholds; in ring oscillators the output is rail-to-rail, so phase noise dominates.

</details>

<details><summary><b>Q14.</b> How does supply noise create phase noise in a ring?</summary>

$K_{VDD} = \partial f/\partial V_{DD}$ turns supply noise into frequency noise: $S_\phi = K_{VDD}^2S_v/f^2$. Rings have large $K_{VDD}$ — why ours is current-fed and FFNC also cancels supply noise.

</details>

<details><summary><b>Q15.</b> Which band contributes most to our PLL's jitter?</summary>

Around the loop bandwidth (≈0.5–5 MHz), where ring noise is no longer suppressed by the loop and still high — Nagam reports ~75% of jitter in 25 kHz–5 MHz for his design.

</details>

<a id='step-23'></a>
## Step 23 — Noise transfer: in-loop sources low-pass and ×N², VCO high-pass; optimum bandwidth

*Roadmap step 23: noise-shaping plots*

### What happens
Every noise source in the loop reaches the output, but through different transfer functions.

### How
With $G$ the loop gain:
$$\Phi_{out} = \underbrace{N\frac{G}{1+G}}_{\text{low-pass, gain }N}(\Phi_{REF}+\Phi_{PD}+\Phi_{div}) + \underbrace{\frac{1}{1+G}}_{\text{high-pass}}\Phi_{VCO}.$$
Reference, PD/CP and divider noise (referred to the PD input) are multiplied by $N$ in amplitude (**+20 log N dB**, +30.1 dB
for $N=32$) inside the bandwidth. The VCO's own noise is suppressed inside the bandwidth and passes outside.

### Why there is an optimum bandwidth
Widening $f_c$ removes more VCO noise but lets more (×N²) in-band noise through. The total is roughly minimised where the
**in-band floor meets the free-running VCO curve**. A ring VCO is very noisy, so it wants a wide loop; a CP-PLL's ×N²
floor stops it — which motivates the SSPLL (Phase 4).

In [ ]:
import sys; sys.path.insert(0, "../models")
import pllmodel as pm
def frame(fc_MHz):
    d = pm.TYPE2_TYP.but(fc=fc_MHz*1e6)
    j, f, L = pm.jitter(d, "type2", band=(10e3, 40e6))
    _, S = pm.spectra(d, "type2", f)
    tr = [dict(x=f, y=10*np.log10(v/2), name=f"{k}: {j[k]*1e12:.2f} ps") for k, v in S.items()]
    tr.append(dict(x=f, y=pm.vco_L(f, d['vco1M'], d['fc3']), name="free-running VCO", line=dict(dash="dot")))
    tr.append(dict(x=f, y=L, name="total", line=dict(width=3, color="black")))
    return tr, f"Type-II CP-PLL, f_c = {fc_MHz} MHz: total jitter {j['TOTAL']*1e12:.2f} ps (10 kHz-40 MHz)"
fig = sweep([0.3, 0.5, 1.0, 1.5, 2.0, 2.5, 3.5, 5.0], frame, "loop bandwidth (MHz)", active=5,
            layout=lambda f: f.update_xaxes(type="log", title="offset (Hz)", range=[3, 8]).update_yaxes(title="dBc/Hz", range=[-150, -60]))
show(fig, 500)

**Try it.** At 0.3 MHz the ring dominates (12 ps); 2.5 MHz gives 2.5 ps and 5 MHz 2.1 ps — but 5 MHz is $f_{REF}/10$, the stability limit, and the ×N² PD/CP floor (≈ −111 dBc/Hz) now dominates in-band. Phases 3 and 4 explain why the SSPLL lowers that floor to −130 dBc/Hz.

### Interview questions — step 23 (10)

<details><summary><b>Q1.</b> Write the output phase noise of a PLL in terms of its sources.</summary>

$S_{out} = |NG/(1+G)|^2(S_{ref}+S_{PD}+S_{div}) + |1/(1+G)|^2 S_{VCO}$ (all referred to their natural points).

</details>

<details><summary><b>Q2.</b> Why is in-band reference noise multiplied by N²?</summary>

Phase is multiplied by N (the loop forces $\phi_{out} = N\phi_{ref}$); power scales as $N^2$, i.e. +20log N dB.

</details>

<details><summary><b>Q3.</b> What is the normalised PD/CP noise floor (FoM_1Hz)?</summary>

$\mathcal L_{1Hz}$: in-band noise normalised as $\mathcal L_{in-band} - 10\log f_{REF} - 20\log N$. Good CP-PLLs reach −220 to −230 dBc/Hz; we assumed −218 typ for SKY130.

</details>

<details><summary><b>Q4.</b> Where is the optimum loop bandwidth?</summary>

Near the offset where the in-band floor (×N²) intersects the free-running VCO noise, adjusted for peaking; then neither source dominates around $f_c$.

</details>

<details><summary><b>Q5.</b> Why does a ring-VCO PLL want a wide bandwidth?</summary>

Ring phase noise is high; the loop suppresses it only below $f_c$, so a wider loop removes more of it.

</details>

<details><summary><b>Q6.</b> Name two things that limit how wide you can make the bandwidth.</summary>

Stability of the sampled loop (≈ $f_{REF}/10$ for a CP-PLL, less with hold delay) and reference spurs/in-band noise passing through.

</details>

<details><summary><b>Q7.</b> How does peaking change integrated jitter?</summary>

Peaking amplifies in-band noise near $f_c$ and VCO noise just above it, adding a hump to the spectrum and to the integral.

</details>

<details><summary><b>Q8.</b> What noise sources are 'VCO-like' (high-passed)?</summary>

Any noise injected at the VCO input or output: VCO intrinsic noise, filter resistor noise (via $K_{VCO}$), supply noise on the VCO.

</details>

<details><summary><b>Q9.</b> If the reference floor is −152 dBc/Hz at 50 MHz, what is its in-band contribution at 1.6 GHz?</summary>

−152 + 20log32 = −121.9 dBc/Hz.

</details>

<details><summary><b>Q10.</b> Why does the proposal still have 2.3 ps jitter without FFNC even with an SSPLL?</summary>

The ring noise outside (and just inside) the 2.5 MHz bandwidth still dominates; lowering the in-band floor helps only up to the loop bandwidth limit.

</details>

<a id='step-24'></a>
## Step 24 — Lab: behavioural PLL in Python and ngspice; check H(s) and lock

*Roadmap step 24: notebook and testbench*

### What you do
1. **Sampled-data simulation.** `sl.cp_pll_sim` simulates a PFD/charge-pump PLL one reference period at a time, with the
   filter and VCO integrated *exactly* between edges (matrix exponential). Compare its phase-step error with the linear
   CT prediction $E(s)=1/(1+G)$ while the bandwidth grows toward $f_{REF}$.
2. **ngspice phase-domain model.** `learn/spice/pll_phase_domain.cir` (controlled sources + real R/C) — run it and
   compare $H(s)$ with Python (already done in step 17: agreement to $10^{-4}$ dB).
3. **Your own Verilog-A PLL.** If you have an ideal Verilog-A PLL, run it with the same $I_p$, $R_1$, $C_1$, $C_2$, $K_{VCO}$, $N$ and
   compare its lock transient with the sampled-data curve below.

In [ ]:
from scipy import signal
Ip, Kv, N, fref = 100e-6, 2*np.pi*300e6, 32, 50e6
def frame(ratio):
    fc = fref*ratio
    d = sl.loop_design_type2(fc, 60, Ip, Kv, N)
    t, err, vc = sl.cp_pll_sim(Ip, d['R'], d['C1'], d['C2'], Kv, N, fref, ncyc=int(12/ratio), phi_step=0.5)
    Ct, Cs = d['C1']+d['C2'], d['C1']*d['C2']/(d['C1']+d['C2'])
    k = Ip*Kv/(2*np.pi*N*Ct)
    den = np.polyadd([d['R']*Cs, 1, 0, 0], [k*d['R']*d['C1'], k])
    _, e = signal.step(signal.lti([d['R']*Cs, 1, 0, 0], den), T=t)
    return ([dict(x=t*1e6, y=err/0.5, name="sampled-data sim (PFD/CP, exact)", mode="lines+markers", marker=dict(size=3)),
             dict(x=t*1e6, y=e, name="linear CT model E(s)", line=dict(dash="dash"))],
            f"f_c = f_REF/{1/ratio:.0f} = {fc/1e6:.2f} MHz: R1 = {d['R']/1e3:.1f} k, C1 = {d['C1']*1e12:.2f} p, C2 = {d['C2']*1e12:.3f} p")
fig = sweep([1/40, 1/20, 1/10, 1/6, 1/4, 1/3], frame, "f_c / f_REF", fmt="{:.3f}", active=1,
            layout=lambda f: f.update_xaxes(title="time (us)").update_yaxes(title="phase error / step", range=[-1.5, 1.5]))
show(fig, 460)

**Try it.** At $f_c = f_{REF}/20$ (our 2.5 MHz) the sampled loop and the CT model overlap. At $f_{REF}/10$ the sampled loop
rings slightly more than predicted; at $f_{REF}/4$ the error swings past −100% of the step; at $f_{REF}/3$ the loop is
unstable (the error grows and clips at the PFD range). The CT model, which ignores that the PFD acts only once per period,
is blind to all of this (**Gardner's limit**, Phase 3 step 37).

**Run the ngspice model yourself:**
```
cd learn/spice && ngspice -b pll_phase_domain.cir      # writes learn/data/pll_phase_domain.csv
```
**Exercise.** Change $I_p$ to 50 µA in both the netlist and the Python cell without re-designing the filter: predict the new
crossover (≈ ×0.6) and PM before running.

### Interview questions — step 24 (10)

<details><summary><b>Q1.</b> Why simulate a PLL in the phase domain instead of transistor level?</summary>

Speed: a 1.6 GHz VCO needs ps time steps for µs of lock (millions of points). Phase-domain models run in milliseconds and expose $H(s)$, stability and noise directly; transistor sims then verify the block gains.

</details>

<details><summary><b>Q2.</b> What does a sampled-data (discrete-time) PLL model capture that the CT model misses?</summary>

That the PFD/CP updates only once per reference period: the effective delay, the staircase control voltage and the resulting instability as $f_c$ approaches $f_{REF}$.

</details>

<details><summary><b>Q3.</b> How do you build a PLL phase-domain model in SPICE?</summary>

Nodes are phases: VCCS $I_p/2\pi\cdot(\phi_{in}-\phi_{div})$ into the real filter; VCO as VCCS $K_{VCO}V_{cont}$ into 1 F; divider as VCVS 1/N; AC analysis gives H(s).

</details>

<details><summary><b>Q4.</b> How do you verify a PLL behavioural model?</summary>

Check DC gain of H = N, crossover and PM against hand calculation, step-response overshoot, and lock from a frequency offset; compare two independent implementations (Python vs SPICE).

</details>

<details><summary><b>Q5.</b> What is the role of Verilog-A in PLL design?</summary>

Behavioural models of blocks (VCO, PFD, CP, divider) that run in Spectre/ngspice (via OpenVAF/OSDI) with transistor-level blocks swapped in one at a time — a mixed-level verification strategy.

</details>

<details><summary><b>Q6.</b> How long does our PLL take to settle after a small phase step?</summary>

With $f_c$ = 2.5 MHz, PM 60°: about 0.5 µs to a few percent, as in the plot.

</details>

<details><summary><b>Q7.</b> Why does the sampled loop diverge from the CT model as $f_c$ approaches $f_{REF}$?</summary>

The loop acts on information that is up to one period old; equivalent to a delay of ~T/2 plus zero-order hold, which costs phase $\approx 180°\,f_c/f_{REF}$ and eventually makes the loop unstable.

</details>

<details><summary><b>Q8.</b> What is a good bandwidth ratio for a CP-PLL and for an SSPLL?</summary>

CP-PLL up to ~$f_{REF}/10$; SSPLL ~$f_{REF}/20$ because of the additional hold delay and spur considerations (Nagam uses $\omega_{REF}/20$).

</details>

<details><summary><b>Q9.</b> What is the exact-discretisation method used in the simulator?</summary>

Between edges the system is LTI with a constant input; the state update is $x_{k+1} = e^{A\Delta t}x_k + \int e^{A\tau}B\,d\tau\cdot u$, computed with an augmented matrix exponential.

</details>

<details><summary><b>Q10.</b> Which parameters of the behavioural model must later come from transistor simulation?</summary>

$K_{VCO}$ and its spread, charge-pump/Gm gain and pulse width, sampler gain (ring amplitude), filter R/C values over corners, and noise of each block.

</details>